# Bank Account Fraud Detection Pipeline

In [82]:
# import required libraries
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [83]:
# define required variable lists

# list of numeric features
numeric_features = [
    'name_email_similarity',
    'credit_risk_score',
    'proposed_credit_limit',
    'intended_balcon_amount',
    'prev_address_months_count',
    'date_of_birth_distinct_emails_4w',
    'current_address_months_count',
    'device_distinct_emails_8w',
    'income',
    'customer_age'
]

# list of catgorical features
categorical_features = [
    'prev_address_months_count_missing',
    'bank_months_count_missing',
    'intended_balcon_amount_missing',
    'foreign_request',
    'housing_status',
    'payment_type',
    'device_os',
    'keep_alive_session',
    'has_other_cards',
    'phone_home_valid',
    'source'
]

# list of columns with missing values.
missing_values_columns = [
    'prev_address_months_count',
    'current_address_months_count',
    'bank_months_count', 
    'device_distinct_emails_8w', 
    'intended_balcon_amount',
    'session_length_in_minutes'
]

missing_values_labels = [
    'prev_address_months_count_missing',
    'current_address_months_count_missing',
    'bank_months_count_missing', 
    'device_distinct_emails_8w_missing', 
    'intended_balcon_amount_missing',
    'session_length_in_minutes_missing'
]

# list of log transform candidate features
log_transform_candidates = [
    'proposed_credit_limit', 
    'intended_balcon_amount', 
    'current_address_months_count', 
    'prev_address_months_count', 
    'device_distinct_emails_8w'
]

# list of features to drop.
drop_features = [
    'intended_balcon_amount', 
    'prev_address_months_count', 
    'housing_status_BG', 
    'payment_type_AE',
    'payment_type_AC', 
    'intended_balcon_amount_missing_1', 
    'prev_address_months_count_missing_1', 
    'bank_months_count_missing_1'
]


In [84]:
# impute missing values
class NegativesToNan(BaseEstimator, TransformerMixin):
    """ This class converts negatives into nan. """
    def __init__(self, columns):
        """ Initialise the class with columns with missing values. """
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform negatives into nan."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
               X[col] = X[col].mask(X[col] < 0, np.nan)
        return X

In [85]:
# create missing value labels
class MissingLabels(BaseEstimator, TransformerMixin):
    """This class creates missing value labels."""
    def __init__(self, columns):
        """Initialise the class with columns with missing values. """
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform missing values by creating new label columns."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[f'{col}_missing'] = (X[col] < 0).astype(int)
        return X

In [86]:
# combine missing value indicators
class CombineMissingLabels(BaseEstimator, TransformerMixin):
    """This class combines missing value labels into one."""
    def __init__(self, columns):
        """Initialise the class with missing value label columns to be combined."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Combine missing labels into one."""
        X = X.copy()
        existing = [col for col in self.columns if col in X.columns]

        if existing:
            X['is_complete'] = X[existing].max(axis=1)

        return X

In [87]:
# log transform all numeric features
class LogTransformer(BaseEstimator, TransformerMixin):
    """This class log transform numeric data."""
    def __init__(self, columns):
        """Initialise the class with columns to be log transformed."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform numeric data."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

In [88]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
    
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        
        # get feature names from ColumnTransformer
        feature_names = self.preprocessor.get_feature_names_out()
        
        # convert numpy array to dataframe
        return pd.DataFrame(X, columns=feature_names)

In [89]:
class ArrayToDataFrame(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        return pd.DataFrame(X, columns=self.columns)

In [90]:
# drop unneeded features
class FeatureDropper(BaseEstimator, TransformerMixin):
    """This class drops unnecessary features."""
    def __init__(self, columns):
        """Initailise with features to drop."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Drop selected features."""
        X = X.copy()
        # drop only columns that exist
        existing = [col for col in self.columns if col in X.columns]
        
        return X.drop(columns=existing)

In [91]:
# data frame feature engineering pipeline
dataframe_pipeline = Pipeline([
    # create missing value labels 
    ('missing_values', MissingLabels(missing_values_columns)),
    # combine missing value indicator
    ('missing_indicator', CombineMissingLabels(missing_values_labels)),
])

In [92]:
# numeric feature engineering pipeline
numeric_features_pipeline = Pipeline([
    # convert negatives to nan
    ('negatives_to_nan', NegativesToNan(missing_values_columns)),
    # impute nan with median
    ('simple_imputer', SimpleImputer(strategy='median')),
    # to dataframe
    ('to_dataframe', ArrayToDataFrame(numeric_features)),
    # log transform numeric features
    ('log_transformer', LogTransformer(log_transform_candidates)),
    # scale all data
    ('scale', StandardScaler())
])

In [93]:
# categorical feature engineering pipeline
categorical_features_pipeline = Pipeline([
    # one hot encoding
    ('one_hot_encoding', OneHotEncoder(
        drop="first",              # prevents dummy variable trap
        handle_unknown="ignore",  # prevents test set errors
        sparse_output=False       # returns dataframe-like output
    ))
])

In [94]:
# build preprocessor with ColumnTransformer
preprocessor = ColumnTransformer([
    # transform numeric features
    ('numeric_features_transformation', numeric_features_pipeline, numeric_features),
    # transform categorical features
    ('categorical_features_transformation', categorical_features_pipeline, categorical_features)
])

In [95]:
# build the logistic regression model pipeline
logistic_regression_pipeline = Pipeline([
    # dataframe engineering pipeline
    ('dataframe_engineering', dataframe_pipeline),
    # preprocessor
    ('preprocessor', preprocessor),
    # dataframe converter
    ('dataframe_maker', DataFrameConverter(preprocessor)),
    # drop features
    ('drop_features', FeatureDropper(drop_features)),
])

Test the pipeline

In [96]:
data = pd.read_csv(r"D:\LZMyDocs\Programme - Financial Crime Risk Analytics\Projects - Financial Crime Risk Analytics\Feedzai Bank Account Fraud Dataset\Feedzai Bank Account Fraud Dataset\Base.csv")

In [97]:
X = data.drop('fraud_bool', axis=1)
y = data['fraud_bool']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Fraud in Train: {y_train.mean():.4%}")
print(f"Fraud in Test: {y_test.mean():.4%}")

Fraud in Train: 1.1029%
Fraud in Test: 1.1030%


In [98]:
# FIT ONLY ON TRAIN DATA (CRITICAL FOR NO LEAKAGE)
X_train_processed = logistic_regression_pipeline.fit_transform(X_train)
# APPLY SAME TRANSFORM TO TEST DATA
X_test_processed = logistic_regression_pipeline.transform(X_test)

AttributeError: Estimator negatives_to_nan does not provide get_feature_names_out. Did you mean to call pipeline[:-1].get_feature_names_out()?